# Metadata curator

This notebook curates a customized dataset based on the archives created with comments.ipynb and metadata.ipynb.
The results are stored in a folder specified by the user below.

In [2]:
import os
import json
import shutil
from zipfile import ZipFile

# Flags

In [1]:
list_file_path = 'D:/JPMETA/extracted/video_ids.json'  # Path to the file containing the ID list
source_meta_dir = 'D:/JPMETA/meta'  # Directory storing id.json files
source_comments_dir = 'D:/JPMETA/comments'  # Directory storing id_threads.json and id_comments.json files
output_zip_path = 'D:/JPMETA/output.zip'  # Final output ZIP file path

# Functions

In [3]:
# Step 1: Read the ID list from the list.json file
def load_id_list(list_file_path):
    with open(list_file_path, 'r', encoding='utf-8') as f:
        id_list = json.load(f)
    return id_list  # This is a list containing multiple IDs

# Step 2: Fuzzy search
def collect_files(id_list, source_meta_dir, source_comments_dir, output_meta_dir, output_comments_dir):
    # Create the target directories
    os.makedirs(output_meta_dir, exist_ok=True)
    os.makedirs(output_comments_dir, exist_ok=True)

    # Search and copy id.json files to the video_meta folder
    for id in id_list:
        meta_file = f"{id}.json"
        source_meta_path = os.path.join(source_meta_dir, meta_file)
        if os.path.exists(source_meta_path):
            shutil.copy(source_meta_path, os.path.join(output_meta_dir, meta_file))

    # Search and fuzzy match id_threads and id_comments files to the video_comments folder
    for root, _, files in os.walk(source_comments_dir):
        for file in files:
            for id in id_list:
                if id in file:
                    # Check if the file is an id_threads or id_comments file
                    if "threads" in file:
                        dest_file = f"{id}_threads.json"
                        shutil.copy(os.path.join(root, file), os.path.join(output_comments_dir, dest_file))
                    elif "comments" in file:
                        dest_file = f"{id}_comments.json"
                        shutil.copy(os.path.join(root, file), os.path.join(output_comments_dir, dest_file))

# Step 3: Package the list.json file, video_meta folder, and video_comments folder into a ZIP file
def create_zip(list_file_path, output_meta_dir, output_comments_dir, output_zip_path):
    with ZipFile(output_zip_path, 'w') as zipf:
        # Add list.json
        zipf.write(list_file_path, os.path.basename(list_file_path))
        
        # Add all files in the video_meta folder
        for root, _, files in os.walk(output_meta_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.join('video_meta', file)  # Path in the ZIP file
                zipf.write(file_path, arcname)
        
        # Add all files in the video_comments folder
        for root, _, files in os.walk(output_comments_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.join('video_comments', file)  # Path in the ZIP file
                zipf.write(file_path, arcname)

# Curate by id list

In [ ]:
# Main program
def curate_by_idlist(list_file_path, source_meta_dir, source_comments_dir, output_zip_path):
    # Create a temporary directory to store the video_meta and video_comments folders
    temp_dir = 'temp_output'
    os.makedirs(temp_dir, exist_ok=True)
    output_meta_dir = os.path.join(temp_dir, 'video_meta')
    output_comments_dir = os.path.join(temp_dir, 'video_comments')

    # Read the ID list
    id_list = load_id_list(list_file_path)

    # Copy files to the video_meta and video_comments folders
    collect_files(id_list, source_meta_dir, source_comments_dir, output_meta_dir, output_comments_dir)

    # Create the ZIP file
    create_zip(list_file_path, output_meta_dir, output_comments_dir, output_zip_path)

    shutil.rmtree(temp_dir)

curate_by_idlist(list_file_path, source_meta_dir, source_comments_dir, output_zip_path)


# Curate by condition

In [ ]:
def matches_conditions(meta_data, conditions):
    """
    Check if a meta_data entry matches all specified conditions.

    :param meta_data: A dictionary representing meta_data of an entry.
    :param conditions: Dictionary of conditions to check against.
    :return: True if all conditions match, False otherwise.
    """
    for key, value in conditions.items():
        if key in meta_data:
            try:
                # Convert numeric strings to integers for comparison
                meta_value = int(meta_data[key]) if isinstance(meta_data[key], str) and meta_data[key].isdigit() else meta_data[key]
                if not isinstance(meta_value, (int, float)):  # Ensure the value is numeric
                    return False
                if meta_value < value:  # Check if the condition is satisfied
                    return False
            except ValueError:
                return False  # Skip entries with invalid numeric values
        else:
            return False  # Condition key not found in meta_data
    return True

def curate_by_condition(source_dir, conditions, output_file):
    """
    Search for IDs in file names within a directory based on specified conditions.

    :param source_dir: Directory containing JSON files to search.
    :param conditions: Dictionary of conditions for filtering (e.g., {"likeCount": 1000}).
    :param output_file: Path to save the filtered ID list.
    """
    filtered_ids = []
    total_files = sum([len(files) for _, _, files in os.walk(source_dir) if files])  # Total number of files
    processed_files = 0  # Counter for processed files

    # Traverse all JSON files in the directory
    for root, _, files in os.walk(source_dir):
        for file in files:
            if file.endswith('.json'):  # Only process JSON files
                # Extract the ID from the file name (e.g., id.json -> id)
                file_id = file.split('.')[0]
                file_path = os.path.join(root, file)

                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        for entry in data:  # Each file may contain multiple entries
                            if "meta_data" in entry:
                                meta_data = entry["meta_data"]
                                # Check if the entry matches all conditions
                                if matches_conditions(meta_data, conditions):
                                    filtered_ids.append(file_id)
                                    print(f"Success: Found matching ID {file_id} in file {file}")
                                    break  # Stop after finding one matching entry in this file
                except (json.JSONDecodeError, KeyError):
                    print(f"Error processing file: {file_path}")

                # Update progress
                processed_files += 1
                print(f"Processed {processed_files}/{total_files} files...", end="\r")

    # Save the filtered IDs to a file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(filtered_ids, f, ensure_ascii=False, indent=4)

    print(f"\n\nFound {len(filtered_ids)} matching IDs. Results saved to {output_file}.")

# Execution Section

In [ ]:
# Specify the directory containing JSON files
source_dir = "D:/JPMETA/meta"

# Specify the output file for filtered IDs
output_file = "D:/JPMETA/filtered_ids.json"

# Manually define search conditions
# Example: Search for entries where "likeCount" > 1000
conditions = {
    "likeCount": 1000,  # Specify the condition: "likeCount" > 1000
    # Uncomment to add more conditions
    # "viewCount": 50000
}

# Run the search function
curate_by_condition(source_dir, conditions, output_file)